<a id="top"></a>
# Marketing bancario: preprocessing con Pipeline di scikit-learn

Questo notebook usa il dataset **Bank Marketing** del repository UCI per costruire un esempio minimale di preprocessing con gli strumenti di scikit-learn. L'obiettivo non è ottimizzare il modello, ma mostrare come trasformare un sottoinsieme di variabili in una matrice di feature pronta per l'addestramento.

<a id="indice"></a>
## Indice

- [Obiettivo e quadro generale](#obiettivo)
- [Caricamento del dataset Bank Marketing](#caricamento)
- [Scelta di un sottoinsieme di variabili](#selezione-sottoinsieme)
- [Configurare scikit-learn per lavorare con DataFrame](#configurazione-scikit)
- [Encoding delle variabili binarie](#binarie)
- [Encoding delle variabili categoriali nominali](#nominali)
- [Encoding delle variabili categoriali ordinali](#ordinali)
- [Trattamento delle variabili numeriche](#numeriche)
- [Costruire un preprocessore unico con ColumnTransformer](#pipeline-preprocessore)
- [Aggiungere un modello di classificazione alla pipeline](#pipeline-modello)
- [Perché salvare pipeline + modello](#sintesi-pipeline)


<a id="obiettivo"></a>
## Obiettivo e quadro generale

Il dataset di marketing bancario contiene variabili eterogenee: numeriche, binarie, categoriali nominali e categoriali ordinali. In questo notebook si lavora su un sottoinsieme ridotto di colonne, scelto per coprire tutti questi casi tipici.

Il percorso segue tre idee semplici:

- definire esplicitamente quali variabili appartengono a ciascun tipo;
- applicare a ogni gruppo un trasformatore di scikit-learn adatto (`OrdinalEncoder`, `OneHotEncoder`, `StandardScaler`, …);
- raccogliere tutto in una singola `Pipeline` che combina preprocessing e modello.

Il risultato finale è un oggetto unico (pipeline + modello) che può essere riutilizzato senza dover riscrivere, a mano, l'intera sequenza di trasformazioni.


<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Nota importante**

Nel dataset originale la colonna `duration` (secondi dell'ultima chiamata) è fortemente predittiva del target, ma non è disponibile nel momento in cui bisogna decidere **chi chiamare**.  
Per evitare leakage, in questo notebook `duration` viene esclusa dal sottoinsieme di variabili usate per il modello.

</div>


<a id="caricamento"></a>
## Caricamento del dataset Bank Marketing

Il dataset viene caricato dal repository UCI tramite la funzione `fetch_ucirepo`.  
Si costruisce un unico `DataFrame` con feature (`X`) e target (`y`), mantenendo i nomi originali delle colonne.


In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

# Caricamento del dataset Bank Marketing (id=222 nel repository UCI)
bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features
y = bank_marketing.data.targets

# Costruiamo un unico DataFrame con feature e target
df = X.copy()
if isinstance(y, pd.DataFrame):
    if y.shape[1] == 1:
        df["y"] = y.iloc[:, 0]
    else:
        for col in y.columns:
            df[col] = y[col]
else:
    df["y"] = y

print("Dimensione feature:", X.shape)
print("Dimensione target:", y.shape)
df.head()

<a id="selezione-sottoinsieme"></a>
## Scelta di un sottoinsieme di variabili

Per rendere più chiaro il lavoro di preprocessing si lavora su un sottoinsieme di colonne che copre i casi più frequenti:

- variabili binarie (`yes` / `no`);
- variabili categoriali nominali (categorie senza ordine intrinseco);
- variabili categoriali ordinali (categorie con un ordine naturale);
- variabili numeriche.

Nel dataset Bank Marketing queste scelte sono:

| Tipo                       | Colonne scelte                                      |
|----------------------------|-----------------------------------------------------|
| Binarie                    | `default`, `housing`                                |
| Categoriali nominali       | `job`, `marital`, `poutcome`                        |
| Categoriali ordinali       | `education`, `month`                                |
| Numeriche                  | `age`, `balance`, `pdays`, `previous`               |

La variabile `poutcome` descrive l'esito della precedente campagna di marketing e viene trattata come categoriale nominale.  
Il target di classificazione resta `y` (sottoscrizione del deposito: `"yes"` / `"no"`).


In [ ]:
import pandas as pd

target_col = "y"

binary_cols = ["default", "housing"]
nominal_cols = ["job", "marital", "poutcome"]
ordinal_cols = ["education", "month"]
numeric_cols = ["age", "balance", "pdays", "previous"]

feature_cols = binary_cols + nominal_cols + ordinal_cols + numeric_cols

df_model = df[feature_cols + [target_col]].copy()
df_model.head()

<a id="configurazione-scikit"></a>
## Configurare scikit-learn per lavorare con DataFrame

Per leggere con chiarezza l'effetto di ogni step di preprocessing conviene chiedere a scikit-learn di restituire `DataFrame` invece di array NumPy anonimi.

Inoltre si prepara uno split train/test: gli oggetti di scikit-learn imparano dai dati di train tramite `fit(…)` e applicano la trasformazione (o la predizione) a nuovi dati tramite `transform(…)` o `predict(…)`.


In [ ]:
from sklearn import set_config
from sklearn.model_selection import train_test_split

set_config(transform_output="pandas")

X = df_model[feature_cols]
y = df_model[target_col].map({"no": 0, "yes": 1}).astype("int8")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.head()

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Attenzione**

Tutti i trasformatori di scikit-learn (encoder, scaler, ecc.) devono essere **fittati solo sui dati di training**.  
Su validation e test si usano esclusivamente i metodi `transform(…)` e `predict(…)` degli oggetti già fittati.  
Mescolare i ruoli porta a leakage e a stime troppo ottimistiche delle prestazioni.

</div>


<a id="binarie"></a>
## Encoding delle variabili binarie

Le variabili `default` e `housing` sono codificate come stringhe `"yes"` / `"no"`.  
Per i modelli numerici è comodo rappresentarle come 0/1, preservando il significato di ordine naturale: 0 = assenza della caratteristica, 1 = presenza.

Si usa `OrdinalEncoder` di scikit-learn, specificando esplicitamente l'ordine delle categorie per ciascuna colonna.


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

binary_encoder = OrdinalEncoder(
    categories=[["no", "yes"], ["no", "yes"]],
    dtype=float,
)

binary_encoder.fit(X_train[binary_cols])

X_train_bin = binary_encoder.transform(X_train[binary_cols])
X_train_bin.head()

<a id="nominali"></a>
## Encoding delle variabili categoriali nominali

Le variabili `job`, `marital` e `poutcome` descrivono categorie senza un ordine naturale.  
In questi casi si usa un encoding one-hot, che produce una colonna per ogni categoria possibile e assegna 0/1 a seconda dell'appartenenza.

Per gestire categorie mai viste in training si imposta `handle_unknown="ignore"`.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

nominal_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)

nominal_encoder.fit(X_train[nominal_cols])

X_train_nom = nominal_encoder.transform(X_train[nominal_cols])
X_train_nom.head()

<a id="ordinali"></a>
## Encoding delle variabili categoriali ordinali

Le variabili `education` e `month` hanno un ordine naturale:

- i livelli di istruzione vanno da `illiterate` a `university.degree`;
- i mesi vanno da gennaio a dicembre.

Per queste colonne ha senso usare un encoding ordinal che rispetti l'ordine logico delle categorie.


In [ ]:
education_order = ["primary", "secondary", "tertiary"]

month_order = [
    "jan", "feb", "mar", "apr", "may", "jun",
    "jul", "aug", "sep", "oct", "nov", "dec",
]

from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder(
    categories=[education_order, month_order],
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    dtype=float,
)

ordinal_encoder.fit(X_train[ordinal_cols])

X_train_ord = ordinal_encoder.transform(X_train[ordinal_cols])
X_train_ord.head()

<a id="numeriche"></a>
## Trattamento delle variabili numeriche

Le variabili numeriche scelte (`age`, `balance`, `pdays`, `previous`) hanno scale molto diverse tra loro.  
Uno step tipico di preprocessing è la standardizzazione: si sottrae la media e si divide per la deviazione standard calcolate sui dati di training.

Per farlo si usa `StandardScaler` di scikit-learn.


In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_scaler = StandardScaler()

numeric_scaler.fit(X_train[numeric_cols])

X_train_num = numeric_scaler.transform(X_train[numeric_cols])
X_train_num.head()

<a id="pipeline-preprocessore"></a>
## Costruire un preprocessore unico con ColumnTransformer

Finora gli step sono stati applicati separatamente, ma in pratica è comodo avere un solo oggetto che conosce:

- quali colonne prendere;
- quale trasformatore applicare a ciascun gruppo.

`ColumnTransformer` permette di combinare più trasformatori in parallelo e restituire una matrice di feature unica.


In [ ]:
from sklearn.compose import ColumnTransformer

preprocess = ColumnTransformer(
    transformers=[
        ("bin", binary_encoder, binary_cols),
        ("ord", ordinal_encoder, ordinal_cols),
        ("nom", nominal_encoder, nominal_cols),
        ("num", numeric_scaler, numeric_cols),
    ]
)

X_train_pre = preprocess.fit_transform(X_train)
X_train_pre.head()

<a id="pipeline-modello"></a>
## Aggiungere un modello di classificazione alla pipeline

Per chiudere il cerchio si aggiunge un modello di classificazione in fondo al preprocessore.  
Si usa una regressione logistica, sufficiente per illustrare il flusso `fit` → `transform` → `predict`.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

clf.fit(X_train, y_train)

y_proba = clf.predict_proba(X_test)[:, 1]
y_pred = clf.predict(X_test)

auc = roc_auc_score(y_test, y_proba)
print(f"ROC AUC (test): {auc:.3f}")
print()
print(classification_report(y_test, y_pred))

In [ ]:
clf

In [ ]:
clf.named_steps['model'].__dict__




<a id="sintesi-pipeline"></a>
## Perché salvare pipeline + modello

La pipeline `clf` contiene sia il modello sia tutte le trasformazioni applicate alle colonne originali:

- mapping delle variabili binarie;
- encoding one-hot delle categoriali nominali;
- encoding ordinal delle categoriali ordinali;
- standardizzazione delle variabili numeriche.

Applicare il modello a nuovi dati grezzi significa passare un `DataFrame` con le stesse colonne di `X_train`:

```python
y_new_proba = clf.predict_proba(df_nuovi_clienti[feature_cols])[:, 1]
```

La logica di preprocessing è riutilizzata automaticamente e resta coerente tra addestramento, validazione e messa in produzione.

Senza una pipeline unica sarebbe necessario ricordare, copiare e tenere allineati tutti i passaggi di trasformazione.  
Con un solo oggetto serializzabile (ad esempio tramite MLflow o `joblib`) si porta con sé sia il modello sia la ricetta completa per preparare i dati.
